In [1]:
import torch
import triton
import triton.language as tl

In [2]:
tl.constexpr

triton.language.core.constexpr

In [17]:
@triton.jit
def matmul_fn(A_p, B_p, C_p, elements, block_size: tl.constexpr):
    pid = tl.program_id(0)

    offsets = pid * block_size + tl.arange(0, block_size)
    mask = offsets < elements

    A = tl.load(A_p + offsets, mask=mask)
    B = tl.load(B_p + offsets, mask=mask)

    C = A * B

    tl.store(C_p + offsets, C, mask=mask)

In [18]:
A = torch.full((4, 4), 3.0, device="cuda", dtype=torch.float32)
B = torch.full((4, 4), 2.0, device="cuda", dtype=torch.float32)
C = torch.empty_like(A)

In [19]:
C

tensor([[0., 2., 0., 2.],
        [0., 2., 0., 2.],
        [0., 2., 0., 2.],
        [0., 2., 0., 2.]], device='cuda:0')

In [20]:
elements = C.numel()

In [21]:
BLOCK = 16

grid = lambda meta: (triton.cdiv(elements, meta["block_size"]),)

matmul_fn[grid](A, B, C, elements, block_size=BLOCK)

print(C)

tensor([[6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.]], device='cuda:0')


In [22]:
print("A:")
print(A)

print("B:")
print(B)

print('-' * 30)
print("C:")
print(C)

A:
tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]], device='cuda:0')
B:
tensor([[2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.]], device='cuda:0')
------------------------------
C:
tensor([[6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.]], device='cuda:0')


In [23]:
import torch
import triton
import triton.language as tl


@triton.jit
def matmul_kernel(
    A_ptr, B_ptr, C_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # offsets do bloco
    offs_m = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    offs_n = pid_n * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    offs_k = tl.arange(0, BLOCK_SIZE)

    # acumulador
    acc = tl.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=tl.float32)

    for k in range(0, K, BLOCK_SIZE):
        a_ptrs = A_ptr + (offs_m[:, None] * stride_am + (k + offs_k)[None, :] * stride_ak)
        b_ptrs = B_ptr + ((k + offs_k)[:, None] * stride_bk + offs_n[None, :] * stride_bn)

        a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & ((k + offs_k)[None, :] < K), other=0.0)
        b = tl.load(b_ptrs, mask=((k + offs_k)[:, None] < K) & (offs_n[None, :] < N), other=0.0)

        acc += tl.dot(a, b)

    # escreve resultado
    c_ptrs = C_ptr + (offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn)
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=mask)


# ------------------------
# TESTE
# ------------------------

M, N, K = 64, 64, 64

A = torch.randn((M, K), device="cuda", dtype=torch.float32)
B = torch.randn((K, N), device="cuda", dtype=torch.float32)
C = torch.empty((M, N), device="cuda", dtype=torch.float32)

BLOCK = 16

grid = lambda meta: (
    triton.cdiv(M, meta["BLOCK_SIZE"]),
    triton.cdiv(N, meta["BLOCK_SIZE"]),
)

matmul_kernel[grid](
    A, B, C,
    M, N, K,
    A.stride(0), A.stride(1),
    B.stride(0), B.stride(1),
    C.stride(0), C.stride(1),
    BLOCK_SIZE=BLOCK
)

# validação
torch_result = A @ B
print(torch.allclose(C, torch_result, atol=1e-2))

True


In [24]:
C

tensor([[ 4.0337e+00,  1.5787e+01,  8.0914e+00,  ..., -1.2574e+01,
         -1.0900e+01,  2.6916e+01],
        [-9.5921e+00, -1.4300e+01,  4.4725e+00,  ..., -7.0875e+00,
          3.7809e-02, -6.5571e+00],
        [-1.9549e-02,  1.7530e+00, -1.4835e+01,  ..., -6.2560e+00,
          2.2092e+00, -4.1393e+00],
        ...,
        [ 5.0566e+00, -7.5998e+00,  3.6012e+00,  ...,  2.9144e+00,
          1.0360e+01, -1.4043e+00],
        [ 4.8589e+00,  6.0036e+00, -6.1307e+00,  ..., -7.7467e+00,
         -2.3297e+00,  3.2879e+00],
        [-5.6710e-02,  1.0847e+01,  3.0114e+00,  ..., -1.7530e+01,
         -3.9110e+00,  1.5297e+01]], device='cuda:0')

In [25]:
torch_result

tensor([[ 4.0337e+00,  1.5787e+01,  8.0914e+00,  ..., -1.2574e+01,
         -1.0900e+01,  2.6916e+01],
        [-9.5921e+00, -1.4300e+01,  4.4725e+00,  ..., -7.0875e+00,
          3.7807e-02, -6.5571e+00],
        [-1.9549e-02,  1.7530e+00, -1.4835e+01,  ..., -6.2560e+00,
          2.2092e+00, -4.1393e+00],
        ...,
        [ 5.0566e+00, -7.5998e+00,  3.6012e+00,  ...,  2.9144e+00,
          1.0360e+01, -1.4043e+00],
        [ 4.8589e+00,  6.0036e+00, -6.1307e+00,  ..., -7.7467e+00,
         -2.3297e+00,  3.2879e+00],
        [-5.6710e-02,  1.0847e+01,  3.0114e+00,  ..., -1.7530e+01,
         -3.9110e+00,  1.5297e+01]], device='cuda:0')